This notebook processes the data gathered from surveys. It first renames all column headers into shorter names so it can be easily referred to during calculations. In addition, it recodes all likert scale answers into integers. 

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata

survey = pd.read_csv("/Users/tonyvo/Desktop/Thesis/survey/raw_survey_33.csv")
df = survey.copy()

# Preliminary Cleanup

In [415]:

# Rename long Google Forms questions into shorter variable names
rename_map = {
    "Timestamp": "timestamp",
    "If you are willing to be contacted for a follow-up study or possible interview, please leave your email address below.\nThis is optional and will only be used in relation to this research project.": "contact_email",
    "Please indicate your age": "age_group",
    "What is the 4-digit postcode of your home address?": "postcode4",
    "How long have you lived in the neighbourhood you currently reside in?": "residence_length",
    "What is your main daily status?": "daily_status",
    "What is your main mode of transport for everyday trips in this area?": "main_mode",
    "How easy is it for you to reach places you use regularly in your daily life from this neighbourhood?": "daily_place_access",
    "How fixed is your daily schedule on most days?": "schedule_fixedness",
    "On a typical day, how much free time do you usually have between finishing your main daily activity and settling at home?": "free_time_after_activity",
    "When you have some free time during or after your daily routine, what do you usually do?": "free_time_activity",
    "Has easier and faster transportation changed where you usually spend time between home, work, school, or other activities?": "transport_changed_timeplace",
    "How easy is it for you to reach places where you might casually spend time, such as cafés, parks, libraries, barbershops, or community spaces?": "casual_place_access",
    "During your day, how often do you pass places where you could easily stop for a bit?": "pass_stop_places_freq",
    "How easy is it for you to stop somewhere for a while on your way home, just because you feel like it?": "spontaneous_stop_ease",
    "If a café, park, or potential third place was a 5-minute detour from your usual route, how likely would you be to stop?": "five_min_detour_likely",
    "How many minutes out of your way would you be willing to go to stop somewhere casually?": "detour_tolerance_mins",
    "What most limits your ability to stop and spend time in local places? ": "main_stop_barrier",
    "Do any of the following make it harder for you to use local places regularly?": "regular_use_barriers",
    "How often do you visit a local third place in this neighbourhood?": "third_place_visit_freq",
    "When you visit these places, is it usually…": "visit_planning_type",
    "How often do you stay longer than originally intended when visiting a local place in your neighbourhood?": "linger_freq",
    "What kinds of places do you use most often for casual time in your neighbourhood?": "place_type_used",
    "What is the main reason you use these places?": "main_place_reason",
    "When you do visit a local place, how often do you return to the same place within the same week?": "same_place_return_freq",
    "Do you have a local place you visit so regularly that staff or other visitors would recognize you?": "recognized_regular",
    "How often do you see familiar faces in local public or semi-public places in your neighbourhood, even if you do not know them personally?": "familiar_faces_freq",
    "How often do you have small interactions with people you recognize in the neighbourhood, such as greeting, nodding, or short conversation?": "small_interactions_freq",
    "I feel at ease in local public places in my neighbourhood.": "ease_public_places",
    "When you spend time in local places, do you tend to be alone, with people you already know, or do you sometimes interact with people you do not know?": "social_use_type",
    "I feel a sense of belonging in this neighbourhood.": "belonging",
    "Being in local places in this neighbourhood makes the area feel more familiar to me.": "place_familiarity",
    "I feel comfortable spending time in local places here without needing a specific purpose.": "comfort_no_purpose",
    "My daily routine leaves me enough unstructured time to stop somewhere locally.": "unstructured_time",
    "Even when places are accessible, I usually feel too rushed to spend time there.": "too_rushed",
    "Improved transport or faster travel in my daily life has made it easier for me to spend time in local places.": "transport_enables_local_time",
    "Improved transport or faster travel in my daily life has made me more likely to go directly home or elsewhere instead of lingering locally.": "transport_encourages_direct_travel",
    "Can you describe a place in this neighbourhood where you feel comfortable spending time? What makes it feel that way?": "comfortable_place_text",
    "Can you describe anything that makes it harder for you to stop, linger, or return regularly to local places in this neighbourhood?": "barriers_text",
    "Do you think transport and daily routines in this area support or undermine casual social life? Please explain briefly.": "transport_routines_social_life_text",
    "Imagine your commute or daily travel became significantly faster. How do you think that would change how you spend time in your neighbourhood?": "faster_travel_effect_text",
    "Can you describe how you first started regularly visiting a local place in your neighbourhood? Was it gradual, accidental, or deliberate?": "regular_visit_origin_text",
}

# Safety check: make sure every raw column has a new name
missing_from_map = set(df.columns) - set(rename_map.keys())
extra_in_map = set(rename_map.keys()) - set(df.columns)

if missing_from_map:
    raise ValueError(f"These columns are missing from rename_map: {missing_from_map}")

if extra_in_map:
    raise ValueError(f"These rename_map keys are not in the CSV: {extra_in_map}")

# Apply renaming
df = df.rename(columns=rename_map)

# Check result
print(df.columns.tolist())
print(df.head())

['timestamp', 'contact_email', 'age_group', 'postcode4', 'residence_length', 'daily_status', 'main_mode', 'daily_place_access', 'schedule_fixedness', 'free_time_after_activity', 'free_time_activity', 'transport_changed_timeplace', 'casual_place_access', 'pass_stop_places_freq', 'spontaneous_stop_ease', 'five_min_detour_likely', 'detour_tolerance_mins', 'main_stop_barrier', 'regular_use_barriers', 'third_place_visit_freq', 'visit_planning_type', 'linger_freq', 'place_type_used', 'main_place_reason', 'same_place_return_freq', 'recognized_regular', 'familiar_faces_freq', 'small_interactions_freq', 'ease_public_places', 'social_use_type', 'belonging', 'place_familiarity', 'comfort_no_purpose', 'unstructured_time', 'too_rushed', 'transport_enables_local_time', 'transport_encourages_direct_travel', 'comfortable_place_text', 'barriers_text', 'transport_routines_social_life_text', 'faster_travel_effect_text', 'regular_visit_origin_text']
            timestamp          contact_email age_group p

In [416]:
# keep only first four digits of all postcodes responses

df['postcode4'] = df['postcode4'].astype(str).str[:4].astype(int)
print(df["postcode4"].astype(str).str.len().value_counts())

postcode4
4    33
Name: count, dtype: int64


In [417]:
def clean_text_value(x): # clean all special and hidden characters like hyphens and extra spaces
    if pd.isna(x):
        return pd.NA
    
    x = unicodedata.normalize("NFKC", str(x)).strip()
    x = re.sub(r"\s+", " ", x)
    
    # normalize all dash types to normal hyphen -
    x = re.sub(r"[‐-‒–—−]", "-", x)
    
    return x

# Apply only to text columns, not numeric columns
text_cols = df.select_dtypes(include=["object", "string"]).columns

df[text_cols] = df[text_cols].apply(lambda col: col.map(clean_text_value))


# Section 2: Your Daily Mobility

In [418]:
# residence_length => "How long have you lived in the neighbourhood you currently reside in?"
#  1 = short residency, 4 = long residency

residence_length_recode = {
    "6 months to 1 year": 1,
    "1-3 years": 2,
    "3-5 years": 3,
    "More than 5 years": 4
}

df["residence_length"] = df["residence_length"].map(residence_length_recode)

print(df["residence_length"].head(20))

0     2
1     1
2     1
3     2
4     3
5     1
6     2
7     1
8     4
9     1
10    1
11    1
12    2
13    2
14    2
15    2
16    2
17    2
18    2
19    2
Name: residence_length, dtype: int64


In [419]:
# daily_place_access =>  "How easy is it for you to reach places you use regularly in your daily life from this neighbourhood?"
# 1 = very easy, 5 = very difficult

difficulty_recode = {
    "Very easy": 1,
    "Easy": 2,
    "Neutral": 3,
    "Difficult": 4,
    "Very difficult": 5
}

df["daily_place_access"] = df["daily_place_access"].map(difficulty_recode)

print(df["daily_place_access"].head(20))

0     1
1     2
2     3
3     1
4     2
5     3
6     1
7     1
8     4
9     1
10    2
11    2
12    2
13    2
14    2
15    1
16    1
17    1
18    4
19    1
Name: daily_place_access, dtype: int64


In [420]:
# schedule_fixedness =>  "How fixed is your daily schedule on most days?"
# 1 = very flexible, 5 = completely fixed

flexibility_recode = {
    "Completely fixed - I follow the same schedule every day": 1,
    "Mostly fixed - small variations but generally predictable": 2,
    "Mixed - some fixed parts, some flexible parts": 3,
    "Mostly flexible - I can usually adjust my schedule": 4,
    "Very flexible - my schedule changes frequently": 5
}

df["schedule_fixedness"] = df["schedule_fixedness"].map(flexibility_recode)

print(df["schedule_fixedness"].head(20))

0     4
1     3
2     4
3     5
4     3
5     3
6     4
7     5
8     3
9     3
10    3
11    3
12    3
13    2
14    3
15    5
16    3
17    2
18    5
19    3
Name: schedule_fixedness, dtype: int64


In [421]:
# free_time_after_activity =>  "On a typical day, how much free time do you usually have between 
# finishing your main daily activity and settling at home?"
# 0 = no time, 5 = a lot of time

free_time_recode = {
    "None - I go straight home": 1,
    "Less than 15 minutes": 2,
    "15-30 minutes": 3,
    "30-60 minutes": 4,
    "More than 1 hour": 5,
    "I do not have a fixed work/study schedule": None
}

df["free_time_after_activity"] = df["free_time_after_activity"].map(free_time_recode)

print(df["free_time_after_activity"].head(20))

0     NaN
1     NaN
2     NaN
3     3.0
4     5.0
5     5.0
6     3.0
7     NaN
8     5.0
9     3.0
10    NaN
11    NaN
12    4.0
13    1.0
14    NaN
15    NaN
16    1.0
17    5.0
18    4.0
19    5.0
Name: free_time_after_activity, dtype: float64


# Section 3: Access to Local Places

In [422]:
# casual_place_access => "How easy is it for you to reach places where you might casually spend time, such as cafés, parks, 
# libraries, barbershops, or community spaces?" 
# 1 = very easy, 5 = very difficult


df["casual_place_access"] = df["casual_place_access"].map(difficulty_recode)

print(df["casual_place_access"].head(20))

0     2
1     1
2     3
3     1
4     2
5     3
6     1
7     1
8     2
9     3
10    1
11    1
12    1
13    1
14    1
15    1
16    1
17    1
18    4
19    2
Name: casual_place_access, dtype: int64


In [423]:
# pass_stop_places_freq =>  "During your day, how often do you pass places where you could easily stop for a bit?" 
# 1 = never, 5 = very often

frequency_recode = {
    "Never": 1,
    "Rarely": 2,
    "Sometimes": 3,
    "Often": 4,
    "Very often": 5
}


df["pass_stop_places_freq"] = df["pass_stop_places_freq"].map(frequency_recode)

print(df["pass_stop_places_freq"].head(20))

0     3
1     4
2     2
3     5
4     4
5     4
6     4
7     5
8     4
9     4
10    5
11    4
12    5
13    3
14    4
15    5
16    3
17    5
18    2
19    5
Name: pass_stop_places_freq, dtype: int64


In [424]:
# spontaneous_stop_ease => "How easy is it for you to stop somewhere for a while on your way home, just because you feel like it?"
# 1 = very easy, 5 = very difficult

df["spontaneous_stop_ease"] = df["spontaneous_stop_ease"].map(difficulty_recode)

print(df["spontaneous_stop_ease"].head(20))


0     3
1     2
2     2
3     2
4     3
5     2
6     3
7     1
8     3
9     4
10    5
11    3
12    1
13    4
14    1
15    2
16    3
17    3
18    4
19    1
Name: spontaneous_stop_ease, dtype: int64


In [ ]:
# five_min_detour_likely => "If a café, park, or potential third place was a 5-minute detour from your usual route, 
# how likely would you be to stop?"
# 1 = very unlikely, 5 = very likely

likelihood_recode = {
    "Very unlikely — I would not detour": 1,
    "Unlikely": 2,
    "Neutral": 3,
    "Likely": 4,
    "Very likely — I would regularly detour for this": 5
}


df["five_min_detour_likely"]=df["five_min_detour_likely"].map(likelihood_recode)

print(df["five_min_detour_likely"].head(20))

0     4.0
1     NaN
2     4.0
3     4.0
4     4.0
5     4.0
6     4.0
7     4.0
8     2.0
9     4.0
10    3.0
11    4.0
12    4.0
13    3.0
14    NaN
15    4.0
16    4.0
17    4.0
18    4.0
19    4.0
Name: five_min_detour_likely, dtype: float64


In [ ]:
print(df["detour_tolerance_mins"].unique())

['5-10 minutes' 'It depends on the day / my mood' '10-15 minutes'
 'More than 15 minutes' '1-5 minutes']


In [427]:
# detour_tolerance_mins => "How many minutes out of your way would you be willing to go to stop somewhere casually?"
# 1 = 0 minutes, 5 = more than 15 minutes

detour_tolerance_recode = {
    "0 minutes — I would not go out of my way": 1,
    "1-5 minutes": 2,
    "5-10 minutes": 3,
    "10-15 minutes": 4,
    "More than 15 minutes": 5,
    "It depends on the day / mood": None
}

df["detour_tolerance_mins"] = df["detour_tolerance_mins"].map(detour_tolerance_recode)

print(df["detour_tolerance_mins"].head(20))


0     3.0
1     3.0
2     NaN
3     NaN
4     4.0
5     4.0
6     5.0
7     NaN
8     NaN
9     2.0
10    NaN
11    4.0
12    NaN
13    NaN
14    3.0
15    3.0
16    NaN
17    4.0
18    3.0
19    4.0
Name: detour_tolerance_mins, dtype: float64


# Section 4: Your Use of Local Places

In [428]:
# third_place_visit_freq => "How often do you visit a local third place in this neighbourhood?"
# 1 = Never, 5 = 3 or more times a week

third_place_visit_freq_recode = {
    "Never": 1,
    "Less than once a month": 2,
    "1-3 times a month": 3,
    "1-2 times a week": 4,
    "3 or more times a week": 5
}

df["third_place_visit_freq"] = df["third_place_visit_freq"].map(third_place_visit_freq_recode)

print(df["third_place_visit_freq"].head(20))

0     4
1     4
2     4
3     3
4     3
5     3
6     4
7     2
8     4
9     3
10    3
11    1
12    3
13    4
14    4
15    4
16    2
17    4
18    1
19    3
Name: third_place_visit_freq, dtype: int64


In [429]:
# visit_planning_type => "When you visit these places, is it usually…"
# 1 = planned, 3 = spontaneous

planning_type_recode = {
    "Planned in advance": 1,
    "Sometimes planned, sometimes spontaneous": 2,
    "Mostly spontaneous": 3
}

df["visit_planning_type"] = df["visit_planning_type"].map(planning_type_recode)

print(df["visit_planning_type"].head(20))

0     1
1     3
2     2
3     2
4     2
5     2
6     2
7     1
8     2
9     3
10    3
11    3
12    3
13    1
14    3
15    2
16    2
17    2
18    1
19    1
Name: visit_planning_type, dtype: int64


In [430]:
# linger_freq => "How often do you stay longer than originally intended when visiting a local place in your neighbourhood?"
# 1 = never, 5 = very often

df["linger_freq"] = df["linger_freq"].map(frequency_recode)
print(df["linger_freq"].head(20))

0     3
1     4
2     3
3     3
4     4
5     3
6     3
7     4
8     2
9     3
10    2
11    2
12    3
13    3
14    5
15    3
16    3
17    5
18    2
19    1
Name: linger_freq, dtype: int64


In [431]:
# same_place_return_freq => "When you do visit a local place, how often do you return to the same place within the same week?"
# # 1 = never, 5 = very often

same_place_return_freq_recode = {
    "Never - I rarely visit the same place twice in a week": 1,
    "Rarely - once in a while": 2,
    "Sometimes - maybe once a week": 3,
    "Often - several times a week": 4,
    "Very often - almost daily": 5
}

df["same_place_return_freq"] = df["same_place_return_freq"].map(same_place_return_freq_recode)
print(df["same_place_return_freq"].head(20))


0     3
1     3
2     2
3     1
4     3
5     3
6     4
7     2
8     4
9     2
10    2
11    2
12    2
13    2
14    3
15    4
16    4
17    3
18    1
19    1
Name: same_place_return_freq, dtype: int64


In [432]:
# recognized_regular => "Do you have a local place you visit so regularly that staff or other visitors would recognize you?"
# 1 = No, 5 = Yes, definitely

yes_no_recode = {
    "No": 1,
    "Probably not": 2,
    "Not sure": 3,
    "Probably yes": 4,
    "Yes, definitely": 5
}

df["recognized_regular"] = df["recognized_regular"].map(yes_no_recode)

print(df["recognized_regular"].head(20))

0     3
1     4
2     4
3     2
4     4
5     3
6     5
7     4
8     5
9     5
10    1
11    4
12    5
13    2
14    2
15    4
16    4
17    4
18    1
19    1
Name: recognized_regular, dtype: int64


# Section 5: Social Experience in Local Places

In [433]:
# familiar_faces_freq => "How often do you see familiar faces in local public or semi-public places in your 
# neighbourhood, even if you do not know them personally?"
# 1 = never, 5 = very often

df["familiar_faces_freq"] = df["familiar_faces_freq"].map(frequency_recode)

print(df["familiar_faces_freq"].head(20))

0     3
1     2
2     3
3     2
4     3
5     3
6     3
7     4
8     5
9     2
10    2
11    2
12    3
13    3
14    5
15    3
16    3
17    3
18    1
19    2
Name: familiar_faces_freq, dtype: int64


In [434]:
# small_interactions_freq => "How often do you have small interactions with people you recognize in the neighbourhood, such as greeting, 
# nodding, or short conversation?"
# 1 = never, 5 = very often

df["small_interactions_freq"] = df["small_interactions_freq"].map(frequency_recode)

print(df["small_interactions_freq"].head(20))

0     1
1     3
2     3
3     1
4     2
5     3
6     2
7     3
8     3
9     2
10    1
11    2
12    5
13    3
14    5
15    3
16    1
17    2
18    1
19    2
Name: small_interactions_freq, dtype: int64


In [435]:
# ease_public_places => "I feel at ease in local public places in my neighbourhood."
# 1 = Strongly disagree, 5 = Strongly agree

agree_recode = {
    "Strongly disagree": 1,
    "Disagree": 2,
    "Neutral": 3,
    "Agree": 4,
    "Strongly agree": 5
}

df["ease_public_places"] = df["ease_public_places"].map(agree_recode)

print(df["ease_public_places"].head(20))


0     4
1     4
2     4
3     4
4     3
5     3
6     2
7     4
8     4
9     4
10    1
11    4
12    5
13    4
14    4
15    4
16    2
17    3
18    4
19    3
Name: ease_public_places, dtype: int64


In [436]:
# social_use_type => "When you spend time in local places, do you tend to be alone, with people you already know, or do you 
# sometimes interact with people you do not know?"
# 1 = zero interaction, 3 = high interaction


interaction_recode = {
    "I am almost always alone": 1,
    "I am usually with people I already know": 1,
    "I sometimes interact with people I do not know": 2,
    "I regularly interact with people I do not know": 3,
    "It varies depending on the place": np.nan
}

df["social_use_type"] = df["social_use_type"].map(interaction_recode)

print(df["social_use_type"].head(20))


0     1.0
1     1.0
2     1.0
3     1.0
4     1.0
5     2.0
6     1.0
7     2.0
8     1.0
9     1.0
10    1.0
11    1.0
12    1.0
13    1.0
14    2.0
15    1.0
16    1.0
17    1.0
18    1.0
19    1.0
Name: social_use_type, dtype: float64


# Section 6: Neighbourhood Belonging

In [437]:
# belonging => "I feel a sense of belonging in this neighbourhood."
# 1 = strongly disagree, 5 = strongly agree

df["belonging"] = df["belonging"].map(agree_recode)
print(df["belonging"].head(20))


0     3
1     3
2     3
3     4
4     3
5     4
6     4
7     4
8     4
9     3
10    2
11    2
12    5
13    3
14    4
15    4
16    4
17    2
18    3
19    3
Name: belonging, dtype: int64


In [438]:
# place_familiarity => "Being in local places in this neighbourhood makes the area feel more familiar to me."
# 1 = strongly disagree, 5 = strongly agree

df["place_familiarity"] = df["place_familiarity"].map(agree_recode)
print(df["place_familiarity"].head(20))


0     4
1     4
2     4
3     4
4     4
5     4
6     4
7     5
8     4
9     4
10    4
11    4
12    5
13    4
14    5
15    4
16    4
17    4
18    4
19    1
Name: place_familiarity, dtype: int64


In [439]:
# comfort_no_purpose => "I feel comfortable spending time in local places here without needing a specific purpose."
# 1 = strongly disagree, 5 = strongly agree

df["comfort_no_purpose"] = df["comfort_no_purpose"].map(agree_recode)
print(df["comfort_no_purpose"].head(20))

0     4
1     3
2     4
3     5
4     3
5     4
6     4
7     2
8     3
9     4
10    2
11    4
12    5
13    4
14    5
15    4
16    3
17    4
18    2
19    2
Name: comfort_no_purpose, dtype: int64


In [440]:
# unstructured_time => "My daily routine leaves me enough unstructured time to stop somewhere locally."
# 1 = strongly disagree, 5 = strongly agree

df["unstructured_time"] = df["unstructured_time"].map(agree_recode)
print(df["unstructured_time"].head(20))

0     4
1     4
2     3
3     4
4     4
5     4
6     4
7     2
8     4
9     3
10    4
11    2
12    5
13    4
14    4
15    4
16    4
17    3
18    2
19    4
Name: unstructured_time, dtype: int64


In [441]:
# too_rushed => "Even when places are accessible, I usually feel too rushed to spend time there."
# 1 = strongly disagree, 5 = strongly agree

df["too_rushed"] = df["too_rushed"].map(agree_recode)
print(df["too_rushed"].head(20))


0     2
1     2
2     2
3     3
4     4
5     4
6     4
7     4
8     2
9     3
10    5
11    4
12    5
13    2
14    2
15    2
16    2
17    4
18    4
19    4
Name: too_rushed, dtype: int64


In [442]:
# transport_enables_local_time => "Improved transport or faster travel in my daily life has made it easier for 
# me to spend time in local places."
# 1 = strongly disagree, 5 = strongly agree

df["transport_enables_local_time"] = df["transport_enables_local_time"].map(agree_recode)
print(df["transport_enables_local_time"].head(20))



0     3
1     3
2     4
3     2
4     4
5     4
6     2
7     5
8     4
9     4
10    2
11    5
12    5
13    3
14    4
15    3
16    3
17    4
18    3
19    3
Name: transport_enables_local_time, dtype: int64


In [443]:
# transport_encourages_direct_travel => Improved transport or faster travel in my daily life has made 
# me more likely to go directly home or elsewhere instead of lingering locally."
# 1 = strongly disagree, 5 = strongly agree

df["transport_encourages_direct_travel"] = df["transport_encourages_direct_travel"].map(agree_recode)
print(df["transport_encourages_direct_travel"].head(20))


0     3
1     4
2     1
3     2
4     4
5     4
6     2
7     4
8     2
9     2
10    5
11    4
12    5
13    3
14    4
15    3
16    4
17    4
18    5
19    5
Name: transport_encourages_direct_travel, dtype: int64


# Final

In [444]:
df.to_csv("survey_recoded.csv", index=False)